**Lab type:** debug  
**Course:** DS105 — Exploratory Data Analysis  
**Lesson:** Understanding Distributions  
**Task:** The AI-generated distribution analysis below runs without errors, but contains 3 bugs. 
For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup: Load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n = 84320

channel = np.random.choice(
    ['web', 'mobile', 'in-store', 'phone'],
    size=n,
    p=[0.45, 0.35, 0.15, 0.05]
)

status = np.random.choice(
    ['delivered', 'pending', 'shipped', 'cancelled', 'processing'],
    size=n,
    p=[0.728, 0.129, 0.097, 0.035, 0.011]
)

# quantity: mostly 1-5, with 14 injected bulk orders
quantity = np.random.choice([1, 2, 3, 4, 5], n, p=[0.4, 0.3, 0.15, 0.1, 0.05])
bulk_idx = np.random.choice(n, 14, replace=False)
quantity[bulk_idx] = np.random.randint(51, 501, 14)

unit_price = np.round(
    np.random.choice([18.99, 29.99, 39.99, 49.99, 69.99, 99.99], n), 2
)

# revenue: right-skewed; cancelled orders have revenue = 0
revenue = np.round(quantity * unit_price * np.random.uniform(0.8, 1.0, n), 2)
revenue[status == 'cancelled'] = 0.0

# shipping_cost: 10% of orders have free shipping (= 0)
shipping_cost = np.round(np.random.uniform(1.5, 25.0, n), 2)
free_idx = np.random.choice(n, int(n * 0.10), replace=False)
shipping_cost[free_idx] = 0.0

df = pd.DataFrame({
    'order_id':      ['ORD-{:05d}'.format(i) for i in range(1, n + 1)],
    'quantity':      quantity,
    'unit_price':    unit_price,
    'revenue':       revenue,
    'shipping_cost': shipping_cost,
    'channel':       channel,
    'status':        status,
})

print(f'Shape: {df.shape}')
print(df[['quantity', 'unit_price', 'revenue', 'shipping_cost']].describe().round(2))

---

## AI-generated distribution analysis

The cells below were produced by an AI assistant given the column names and a prompt to 
"analyse the distribution of each numeric column". The code runs without errors. 
Your job is to find the 3 bugs.

### Step 1 — Histogram overview

In [ ]:
# AI-generated
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, col in zip(axes, ['quantity', 'unit_price', 'revenue', 'shipping_cost']):
    df[col].hist(bins=30, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

The histograms show that `revenue` and `shipping_cost` are right-skewed 
(long tail to the right). The AI next applies a log transformation to reduce the skew.

### Step 2 — Log transformation for right-skewed columns

In [ ]:
# AI-generated  ← contains Bug 1
df['log_revenue'] = np.log(df['revenue'])           # log-transform revenue
df['log_shipping'] = np.log(df['shipping_cost'])    # log-transform shipping_cost

print('Revenue (original):')
print(f"  mean: {df['revenue'].mean():.2f}  median: {df['revenue'].median():.2f}")

print('\nLog revenue:')
print(f"  mean: {df['log_revenue'].mean():.4f}  median: {df['log_revenue'].median():.4f}")

print('\nLog shipping_cost:')
print(f"  mean: {df['log_shipping'].mean():.4f}  median: {df['log_shipping'].median():.4f}")

**What to find:** The log-transformed columns report `nan` or `-inf` for the mean. 
Look at the transform applied — is it safe when the column contains zeros?

**Hint:** Check `df['revenue'].min()` and `df['shipping_cost'].min()`. 
What does `np.log(0)` return? What should be used instead?

In [ ]:
# Detect: inspect the minimum values and the problematic outputs
print('revenue min:', df['revenue'].min())
print('shipping_cost min:', df['shipping_cost'].min())
print('log(0):', np.log(0))

# Fix: replace np.log with np.log1p — log1p(x) = log(1 + x), safe when x >= 0
# YOUR CODE HERE

**Explanation:** Write your answer here — why does `np.log` fail for these columns, and what does `np.log1p` do differently?

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What the bug is:** `np.log(0)` returns `-inf` (negative infinity), and `np.log` of any non-positive value produces `-inf` or `nan`. Both `revenue` and `shipping_cost` contain zeros, so applying `np.log` directly produces `-inf` values that propagate into the mean, reporting `nan`.

**Why it causes wrong behaviour:** The resulting columns contain invalid values that break any downstream statistics or model training. The mean reports `nan` because arithmetic with `-inf` is undefined in the expected sense.

**Correct approach:** Use `np.log1p(x)`, which computes `log(1 + x)`. At `x = 0` this returns `log(1) = 0` — numerically safe and mathematically meaningful for non-negative columns that include zeros.

</details>

### Step 3 — Outlier investigation

In [ ]:
# AI-generated  ← contains Bug 2
print(df['quantity'].describe())

# Check for extreme outliers
n_outliers = (df['quantity'] > 500).sum()
print(f'\nOrders with quantity > 500: {n_outliers}')
print('Conclusion: no outliers found in quantity.')

**What to find:** The AI concludes "no outliers found", but the `describe()` output above 
tells a different story. Look at the gap between the 75th percentile and the max — 
is the threshold `> 500` the right place to look?

**Hint:** The max is 500. What threshold makes the bulk orders visible? 
Try inspecting the tail of the distribution.

In [ ]:
# Detect: look at the distribution tail
print(df['quantity'].describe())
print()

# How many values are beyond the 99th percentile?
p99 = df['quantity'].quantile(0.99)
print(f'99th percentile: {p99}')

# Fix: use a threshold that actually finds the bulk orders
# YOUR CODE HERE — find the rows and inspect them

**Explanation:** Write your answer here — why does using `> 500` (the max) as a threshold find nothing, and what approach should you use to identify outliers worth investigating?

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What the bug is:** The outlier threshold is set at `> 500`, which is exactly the maximum value in `quantity`. Since no value can exceed the column maximum, the condition is guaranteed to return zero rows by construction.

**Why it causes wrong behaviour:** The AI concludes "no outliers found" when the column clearly contains a bimodal distribution — most retail/online orders have 1–7 units, while wholesale/enterprise orders have 10–500 units. The extreme tail is real and worth investigating.

**Correct approach:** Use a data-driven threshold: `p99 = df['quantity'].quantile(0.99)`. Rows above the 99th percentile represent the extreme tail regardless of what the absolute maximum is.

</details>

### Step 4 — Revenue distribution by channel

In [ ]:
# AI-generated  ← contains Bug 3
sns.kdeplot(data=df, x='revenue', hue='channel')
plt.title('Revenue distribution by channel')
plt.show()

**What to find:** The KDE plot looks reasonable for `web` and `mobile`, but the `phone` 
channel (5% of orders) is barely visible — its curve is squashed flat compared to the others. 
Is that because phone orders have unusually concentrated revenue, or is it the plot?

**Hint:** Check the default value of the `common_norm` parameter in `sns.kdeplot`. 
What does it do, and how should you set it when comparing groups of different sizes?

In [ ]:
# Verify: how many rows per channel?
print(df['channel'].value_counts())

# Fix: add the parameter that normalises each group independently
# YOUR CODE HERE

**Explanation:** Write your answer here — what does `common_norm=True` (the default) do to small groups, and why does `common_norm=False` give you a more useful comparison?

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What the bug is:** `sns.kdeplot` defaults to `common_norm=True`, which normalises all group curves so their combined area across the whole dataset sums to 1. A group that represents only 5% of the data (the `phone` channel) receives only 5% of that area budget — its curve is squashed near-zero regardless of how concentrated its distribution is.

**Why it causes wrong behaviour:** The phone channel's curve appears essentially flat, which might suggest its revenue is highly dispersed — when in fact the curve is just artificially scaled down due to the small group share.

**Correct approach:** Set `common_norm=False`. This normalises each group's curve independently so every channel integrates to 1, making the shape of each distribution directly comparable at equal visual scale.

</details>

---

## Corrected analysis

Run the cells below to see the distribution analysis with all three bugs fixed.

In [ ]:
# Corrected log transformation
df['log_revenue'] = np.log1p(df['revenue'])
df['log_shipping'] = np.log1p(df['shipping_cost'])

print('Log revenue:')
print(f"  mean: {df['log_revenue'].mean():.4f}  median: {df['log_revenue'].median():.4f}")
print('Log shipping_cost:')
print(f"  mean: {df['log_shipping'].mean():.4f}  median: {df['log_shipping'].median():.4f}")

In [ ]:
# Corrected outlier investigation
p99 = df['quantity'].quantile(0.99)
bulk = df[df['quantity'] > p99]
print(f'Rows above 99th percentile ({p99:.0f}): {len(bulk)}')
print(bulk[['order_id', 'quantity', 'unit_price', 'revenue', 'channel']].to_string(index=False))

In [ ]:
# Corrected KDE — normalised per group
sns.kdeplot(data=df, x='revenue', hue='channel', common_norm=False)
plt.title('Revenue distribution by channel (each group normalised independently)')
plt.show()

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Bug 1 — np.log on zeros:** `np.log(0) = -inf`; use `np.log1p()` for non-negative columns that may contain zeros.

2. **Bug 2 — threshold at max:** Using `> max_value` guarantees zero results; use a quantile-based threshold (e.g. 99th percentile) instead.

3. **Bug 3 — common_norm=True:** The default normalises curves across the whole dataset, squashing small groups; set `common_norm=False` to compare shapes per group.

</details>